Прогнозирование коэффициентов текучести среди персонала (U) и среди сотрудников-стажеров (Q)

In [4]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

In [5]:
file_path = "/content/data_hist.xlsx"

hist = pd.read_excel(file_path, sheet_name="historical_2023_2025")
forecast_2026 = pd.read_excel(file_path, sheet_name="forecast_2026")

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


Для учета динамики текучести были сформированы лаги: 1 - значение коэффициента за прошлый месяц, 2 - за два месяца назад, 3 - среднее значение за последние три месяца. Для учета не только календарного месяца и должность, но и предыдущую динамику.

In [7]:
# Лаги для U
hist["U_lag_1"] = hist.groupby("vacancy")["U_staff_attrition"].shift(1)
hist["U_lag_2"] = hist.groupby("vacancy")["U_staff_attrition"].shift(2)
hist["U_roll_3"] = (hist.groupby("vacancy")["U_staff_attrition"].transform(lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()))

# Лаги для Q
hist["Q_lag_1"] = hist.groupby("vacancy")["Q_intern_attrition"].shift(1)
hist["Q_lag_2"] = hist.groupby("vacancy")["Q_intern_attrition"].shift(2)
hist["Q_roll_3"] = ( hist.groupby("vacancy")["Q_intern_attrition"].transform(lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()))
hist.head(15)

,date,year,month,vacancy,has_internship,staff_plan,staff_fact,staff_gap,U_staff_attrition,Q_intern_attrition,...,K_stage1_conversion,hire_fact,contacts_fact,candidates_fact,U_lag_1,U_lag_2,U_roll_3,Q_lag_1,Q_lag_2,Q_roll_3
0,2023-01,2023,1,Инспектор контроля,True,520,500,20,0.0201,0.0940,...,0.4289,18,168,392,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-02,2023,2,Инспектор контроля,True,520,502,18,0.0176,0.0775,...,0.4301,15,137,319,0.0201,NaN,0.020100,0.0940,NaN,0.094000
2,2023-03,2023,3,Инспектор контроля,True,520,506,14,0.0174,0.0831,...,0.4332,16,164,379,0.0176,0.0201,0.018850,0.0775,0.0940,0.085750
3,2023-04,2023,4,Инспектор контроля,True,520,508,12,0.0221,0.1069,...,0.4294,10,101,235,0.0174,0.0176,0.018367,0.0831,0.0775,0.084867
4,2023-05,2023,5,Инспектор контроля,True,521,501,20,0.0269,0.1149,...,0.4378,19,194,443,0.0221,0.0174,0.019033,0.1069,0.0831,0.089167
5,2023-06,2023,6,Инспектор контроля,True,521,498,23,0.0281,0.1168,...,0.4152,20,193,465,0.0269,0.0221,0.022133,0.1149,0.1069,0.101633
6,2023-07,2023,7,Инспектор контроля,True,521,500,21,0.0281,0.1305,...,0.4289,20,198,462,0.0281,0.0269,0.025700,0.1168,0.1149,0.112867
7,2023-08,2023,8,Инспектор контроля,True,521,503,18,0.0259,0.1162,...,0.4018,21,198,493,0.0281,0.0281,0.027700,0.1305,0.1168,0.120733
8,2023-09,2023,9,Инспектор контроля,True,521,502,19,0.0215,0.1256,...,0.3859,18,186,482,0.0259,0.0281,0.027367,0.1162,0.1305,0.121167
9,2023-10,2023,10,Инспектор контроля,True,521,505,16,0.0226,0.1129,...,0.4478,17,161,360,0.0215,0.0259,0.025167,0.1256,0.1162,0.124100


Обучение моделей на данных 23-24 годов + проверки результатов на 25

In [10]:
train = hist[hist["year"] <= 2024].copy()
test = hist[hist["year"] == 2025].copy()

print(train.shape)
print(test.shape)

(120, 31)
(60, 31)


In [11]:
hist["planned_start_groups"] = hist["start_groups"]
train["planned_start_groups"] = train["start_groups"]
test["planned_start_groups"] = test["start_groups"]

target_u = "U_staff_attrition"
features_u = ["month", "vacancy", "has_internship", "staff_plan",
             "planned_start_groups", "group_limit", "internship_capacity",
             "contact_lag_days", "max_hire_monthly", "K_total_conversion",
             "K_stage1_conversion", "U_lag_1", "U_lag_2", "U_roll_3",]


target_q = "Q_intern_attrition"
features_q = ["month", "vacancy", "staff_plan", "planned_start_groups",
              "group_limit", "internship_capacity", "contact_lag_days",
              "K_total_conversion", "K_stage1_conversion",
              "Q_lag_1", "Q_lag_2", "Q_roll_3",]

In [13]:
X_train_u = train[features_u].copy()
y_train_u = train[target_u].copy()

X_test_u = test[features_u].copy()
y_test_u = test[target_u].copy()

print("U train:", X_train_u.shape, y_train_u.shape)
print("U test:", X_test_u.shape, y_test_u.shape)

U train: (120, 14) (120,)
U test: (60, 14) (60,)


In [14]:
train_q = train[(train["has_internship"] == True) & (train[target_q].notna())].copy()
test_q = test[(test["has_internship"] == True) & (test[target_q].notna())].copy()

X_train_q = train_q[features_q].copy()
y_train_q = train_q[target_q].copy()

X_test_q = test_q[features_q].copy()
y_test_q = test_q[target_q].copy()

print("Q train:", X_train_q.shape, y_train_q.shape)
print("Q test:", X_test_q.shape, y_test_q.shape)

Q train: (48, 12) (48,)
Q test: (24, 12) (24,)


In [16]:
def evaluate_models_for_target(
    X_train, y_train,
    X_test, y_test,
    features,
    categorical_features,
    lag_baseline_col=None,
    target_name="target"):

    numeric_features = [col for col in features if col not in categorical_features]
    numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")),])
    categorical_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore")),])
    preprocessor = ColumnTransformer(transformers=[("num", numeric_transformer, numeric_features), ("cat", categorical_transformer, categorical_features),])

    models = {
        "Ridge Regression": Ridge(alpha=1.0),

        "Random Forest": RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            min_samples_leaf=2),

        "CatBoost": CatBoostRegressor(
            iterations=300,
            learning_rate=0.03,
            depth=3,
            loss_function="RMSE",
            random_seed=42,
            verbose=False),

        "LightGBM": LGBMRegressor(
            n_estimators=150,
            learning_rate=0.03,
            max_depth=3,
            num_leaves=7,
            min_child_samples=5,
            random_state=42,
            verbose=-1),}

    rows = []
    if lag_baseline_col is not None and lag_baseline_col in X_test.columns:
        baseline_pred = X_test[lag_baseline_col].fillna(y_train.mean())

        rows.append({
            "target": target_name,
            "model": "Last month baseline",
            "MAE": mean_absolute_error(y_test, baseline_pred),
            "RMSE": mean_squared_error(y_test, baseline_pred) ** 0.5,
            "R2": r2_score(y_test, baseline_pred)})

    for model_name, model in models.items():
        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)])

        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)

        rows.append({
            "target": target_name,
            "model": model_name,
            "MAE": mean_absolute_error(y_test, pred),
            "RMSE": mean_squared_error(y_test, pred) ** 0.5,
            "R2": r2_score(y_test, pred)})

    result = pd.DataFrame(rows)

    result["MAE_pct_points"] = result["MAE"] * 100
    result["RMSE_pct_points"] = result["RMSE"] * 100
    result = result.sort_values("MAE").reset_index(drop=True)

    return result

In [19]:
categorical_features_u = ["vacancy", "has_internship"]

comparison_u = evaluate_models_for_target(
    X_train=X_train_u,
    y_train=y_train_u,
    X_test=X_test_u,
    y_test=y_test_u,
    features=features_u,
    categorical_features=categorical_features_u,
    lag_baseline_col="U_lag_1",
    target_name="U_staff_attrition")


comparison_u[[ "target", "model", "R2", "MAE_pct_points", "RMSE_pct_points"]]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,target,model,R2,MAE_pct_points,RMSE_pct_points
0,U_staff_attrition,CatBoost,0.889208,0.214925,0.279150
1,U_staff_attrition,LightGBM,0.851252,0.249519,0.323451
2,U_staff_attrition,Random Forest,0.824881,0.259300,0.350953
3,U_staff_attrition,Last month baseline,0.776121,0.319000,0.396816
4,U_staff_attrition,Ridge Regression,0.538851,0.457278,0.569513


По результатам backtest на 2025 годе для прогноза текучести основного штата (U) наилучшее качество показала модель CatBoost.

Она имеет минимальные значения MAE (0,215) и RMSE (0,279). Она также показала наибольшее значение R² (0,889), то есть лучше остальных моделей объясняет вариацию коэффициента текучести.


In [20]:
categorical_features_q = ["vacancy"]

comparison_q = evaluate_models_for_target(
    X_train=X_train_q,
    y_train=y_train_q,
    X_test=X_test_q,
    y_test=y_test_q,
    features=features_q,
    categorical_features=categorical_features_q,
    lag_baseline_col="Q_lag_1",
    target_name="Q_intern_attrition")

comparison_q[[ "target", "model", "R2", "MAE_pct_points", "RMSE_pct_points"]]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,target,model,R2,MAE_pct_points,RMSE_pct_points
0,Q_intern_attrition,LightGBM,0.866266,0.428232,0.550040
1,Q_intern_attrition,CatBoost,0.869236,0.461506,0.543898
2,Q_intern_attrition,Random Forest,0.806658,0.585347,0.661358
3,Q_intern_attrition,Ridge Regression,0.668425,0.722279,0.866092
4,Q_intern_attrition,Last month baseline,0.577857,0.781250,0.977243


По результатам backtest на 2025 годе для прогноза выбытия сотрудников-стажеров (Q) наименьшую MAE показала модель LightGBM - 0,428. Это означает, что в среднем прогноз коэффициента выбытия стажеров отклоняется от фактического значения менее чем на 0,5%.

Реализация прогноза для 2026 года

In [21]:
best_u_name = "CatBoost"
best_q_name = "LightGBM"

def get_model_by_name(model_name):
    if model_name == "Ridge Regression":
        return Ridge(alpha=1.0)

    if model_name == "Random Forest":
        return RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            min_samples_leaf=2)

    if model_name == "CatBoost":
        return CatBoostRegressor(
            iterations=300,
            learning_rate=0.03,
            depth=3,
            loss_function="RMSE",
            random_seed=42,
            verbose=False)

    if model_name == "LightGBM":
        return LGBMRegressor(
            n_estimators=150,
            learning_rate=0.03,
            max_depth=3,
            num_leaves=7,
            min_child_samples=5,
            random_state=42,
            verbose=-1)

In [23]:
best_model_u = get_model_by_name(best_u_name)

numeric_features_u = [col for col in features_u if col not in categorical_features_u]
numeric_transformer_u = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")),])
categorical_transformer_u = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore")),])

preprocessor_u = ColumnTransformer(transformers=[("num", numeric_transformer_u, numeric_features_u), ("cat", categorical_transformer_u, categorical_features_u),])

final_model_u = Pipeline(steps=[
    ("preprocessor", preprocessor_u),
    ("model", best_model_u)])

X_hist_u = hist[features_u].copy()
y_hist_u = hist[target_u].copy()

final_model_u.fit(X_hist_u, y_hist_u)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['month', 'staff_plan',
                                                   'planned_start_groups',
                                                   'group_limit',
                                                   'internship_capacity',
                                                   'contact_lag_days',
                                                   'max_hire_monthly',
                                                   'K_total_conversion',
                                                   'K_stage1_conversion',
                                                   'U_lag_1', 'U_lag_2',
                                                   'U_roll_3']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['vacancy',
                                                   'has_internship'])])),
                ('model',
                 CatBoostRegressor(depth=3, iterations=300, learning_rate=0.03, loss_function='RMSE', random_seed=42, verbose=False))])

In [24]:
best_model_q = get_model_by_name(best_q_name)

hist_q = hist[(hist["has_internship"] == True) & (hist[target_q].notna())].copy()

numeric_features_q = [col for col in features_q if col not in categorical_features_q]
numeric_transformer_q = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")),])
categorical_transformer_q = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore")),])

preprocessor_q = ColumnTransformer(transformers=[("num", numeric_transformer_q, numeric_features_q), ("cat", categorical_transformer_q, categorical_features_q),])

final_model_q = Pipeline(steps=[
    ("preprocessor", preprocessor_q),
    ("model", best_model_q)])

X_hist_q = hist_q[features_q].copy()
y_hist_q = hist_q[target_q].copy()

final_model_q.fit(X_hist_q, y_hist_q)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['month', 'staff_plan',
                                                   'planned_start_groups',
                                                   'group_limit',
                                                   'internship_capacity',
                                                   'contact_lag_days',
                                                   'K_total_conversion',
                                                   'K_stage1_conversion',
                                                   'Q_lag_1', 'Q_lag_2',
                                                   'Q_roll_3']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['vacancy'])])),
                ('model',
                 LGBMRegressor(learning_rate=0.03, max_depth=3,
                               min_child_samples=5, n_estimators=150,
                               num_leaves=7, random_state=42, verbose=-1))])

Приведение в порядок полученных значений, чистка относитально данных конца 2025 года и сохранение итогового прогноза для 2026 года


In [33]:
hist_for_work = hist.copy()
hist_for_work["U_pred"] = hist_for_work["U_staff_attrition"]
hist_for_work["Q_pred"] = hist_for_work["Q_intern_attrition"]


forecast_2026_ml = forecast_2026.copy()
forecast_2026_ml["K_total_conversion"] = forecast_2026_ml["K_total_expected"]
forecast_2026_ml["K_stage1_conversion"] = forecast_2026_ml["K_stage1_expected"]
forecast_2026_ml["U_staff_attrition"] = np.nan
forecast_2026_ml["Q_intern_attrition"] = np.nan
forecast_2026_ml["U_pred"] = np.nan
forecast_2026_ml["Q_pred"] = np.nan

# Объединение все в 1 набор
work_df = pd.concat([hist_for_work, forecast_2026_ml],ignore_index=True, sort=False)
work_df = work_df.sort_values(["vacancy", "year", "month"]).reset_index(drop=True)
work_df

,date,year,month,vacancy,has_internship,staff_plan,staff_fact,staff_gap,U_staff_attrition,Q_intern_attrition,...,Q_lag_2,Q_roll_3,planned_start_groups,U_pred,Q_pred,initial_staff_fact_2025_12,initial_interns_2025_12,internship_duration_months,K_total_expected,K_stage1_expected
0,2023-01,2023,1,Инспектор безопасности,True,230,221.0,9.0,0.0123,0.0746,...,NaN,NaN,1,0.0123,0.0746,NaN,NaN,NaN,NaN,NaN
1,2023-02,2023,2,Инспектор безопасности,True,230,224.0,6.0,0.0138,0.0724,...,NaN,0.074600,1,0.0138,0.0724,NaN,NaN,NaN,NaN,NaN
2,2023-03,2023,3,Инспектор безопасности,True,230,226.0,4.0,0.0189,0.0751,...,0.0746,0.073500,1,0.0189,0.0751,NaN,NaN,NaN,NaN,NaN
3,2023-04,2023,4,Инспектор безопасности,True,235,225.0,10.0,0.0192,0.0791,...,0.0724,0.074033,2,0.0192,0.0791,NaN,NaN,NaN,NaN,NaN
4,2023-05,2023,5,Инспектор безопасности,True,235,224.0,11.0,0.0201,0.0857,...,0.0751,0.075533,2,0.0201,0.0857,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,2026-08,2026,8,Специалист обслуживания,False,150,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0.0,0.095,0.4
236,2026-09,2026,9,Специалист обслуживания,False,150,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0.0,0.095,0.4
237,2026-10,2026,10,Специалист обслуживания,False,150,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0.0,0.095,0.4
238,2026-11,2026,11,Специалист обслуживания,False,150,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0.0,0.095,0.4


Функция для расчета лагов спрогнозированных длдя 2026 года значений

In [27]:
def update_lags_for_row(work_df, idx, pred_col, prefix):
    vacancy = work_df.loc[idx, "vacancy"]
    year = work_df.loc[idx, "year"]
    month = work_df.loc[idx, "month"]

    prev_rows = work_df[(work_df["vacancy"] == vacancy) & ((work_df["year"] < year) |
        ((work_df["year"] == year) & (work_df["month"] < month)))].sort_values(["year", "month"])

    values = prev_rows[pred_col].dropna().tail(3).to_list()
    if len(values) >= 1:
        work_df.loc[idx, f"{prefix}_lag_1"] = values[-1]

    if len(values) >= 2:
        work_df.loc[idx, f"{prefix}_lag_2"] = values[-2]

    if len(values) >= 1:
        work_df.loc[idx, f"{prefix}_roll_3"] = np.mean(values)

    return work_df

In [28]:
for current_month in sorted(work_df.loc[work_df["year"] == 2026, "month"].unique()):
    month_mask = (work_df["year"] == 2026) & (work_df["month"] == current_month)
    month_indices = work_df[month_mask].index

    for idx in month_indices:
        work_df = update_lags_for_row(
            work_df=work_df,
            idx=idx,
            pred_col="U_pred",
            prefix="U")

        X_row_u = work_df.loc[[idx], features_u]
        u_pred = final_model_u.predict(X_row_u)[0]
        work_df.loc[idx, "U_pred"] = u_pred


        if work_df.loc[idx, "has_internship"] == True:
            work_df = update_lags_for_row(
                work_df=work_df,
                idx=idx,
                pred_col="Q_pred",
                prefix="Q")

            X_row_q = work_df.loc[[idx], features_q]
            q_pred = final_model_q.predict(X_row_q)[0]
            work_df.loc[idx, "Q_pred"] = q_pred
        else:
            work_df.loc[idx, "Q_pred"] = np.nan

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

In [29]:
ml_forecast_2026 = work_df[work_df["year"] == 2026][[
    "date",
    "year",
    "month",
    "vacancy",
    "has_internship",
    "staff_plan",
    "U_pred",
    "Q_pred",
    "K_total_expected",
    "K_stage1_expected"]].copy()

ml_forecast_2026["U_pred_pct"] = ml_forecast_2026["U_pred"] * 100
ml_forecast_2026["Q_pred_pct"] = ml_forecast_2026["Q_pred"] * 100

ml_forecast_2026.head(20)

,date,year,month,vacancy,has_internship,staff_plan,U_pred,Q_pred,K_total_expected,K_stage1_expected,U_pred_pct,Q_pred_pct
36,2026-01,2026,1,Инспектор безопасности,True,248,0.018346,0.085375,0.120,0.46,1.834569,8.537470
37,2026-02,2026,2,Инспектор безопасности,True,248,0.017744,0.084659,0.120,0.46,1.774377,8.465920
38,2026-03,2026,3,Инспектор безопасности,True,248,0.020580,0.082283,0.120,0.46,2.058012,8.228312
39,2026-04,2026,4,Инспектор безопасности,True,248,0.022641,0.091598,0.120,0.46,2.264122,9.159772
40,2026-05,2026,5,Инспектор безопасности,True,248,0.023698,0.097106,0.120,0.46,2.369828,9.710637
41,2026-06,2026,6,Инспектор безопасности,True,248,0.024938,0.098228,0.120,0.46,2.493832,9.822795
42,2026-07,2026,7,Инспектор безопасности,True,249,0.025303,0.103118,0.120,0.46,2.530297,10.311834
43,2026-08,2026,8,Инспектор безопасности,True,249,0.025564,0.106447,0.120,0.46,2.556376,10.644719
44,2026-09,2026,9,Инспектор безопасности,True,249,0.022959,0.108731,0.120,0.46,2.295873,10.873102
45,2026-10,2026,10,Инспектор безопасности,True,249,0.021656,0.100376,0.120,0.46,2.165556,10.037608


Загрузка итогового набора данных для 2026 года


In [31]:
ml_forecast_2026.to_excel("forecast_u_q_2026.xlsx",index=False)